# agent불러오기 연습

In [ ]:
# import os
# # 원래 이렇게 그냥 실행시키면 안됨. vscode가 ipynb확장다운받으면서 자동으로 해당 api key가져올 수 있게 세팅된거임
# # 아래 패키지가 현재 폴더 .env 파일에서 환경변수를 로드해주는 역할을 함
# from dotenv import load_dotenv
# API 키 잘 가지고 오는지 확인만 해보는 셀
# 이후에는 사용하지 않을 거임
# print(os.getenv("OPENAI_API_KEY"))

[model인자에서 어떤 모델 쓰는지 리스트 링크](https://platform.openai.com/docs/pricing)

In [ ]:
import openai

client = openai.OpenAI()

response = client.chat.completions.create(
    model =  "gpt-4o-mini", # 대화에 사용할 모델 선택
    n = 10, # 답변 여러개 받고 싶으면 n 조절, 기본값 1
    messages= [      
               {
                'role': 'user'  ,
                'content' : 'What is the copital of Korea?' # 모델에 질문할 것
               }
    ]
)
response
'''받는 답변은 chat completion 객체로, 받은 답변을 파이썬에서 쓸 수 있게 변환해주는 것
여기서 필요한 건 choices. 이건 리스트 형태로 되어 있음. 우리가 사용하는 모델은 그저
질문에 대한 가장 그럴듯한 답변은 무엇일까?에 대한 확률이 제일 높은 답을 찾음'''

ChatCompletion(id='chatcmpl-Ci1BOvIQwsccB3IGwGp124EmPCqdn', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of South Korea is Seoul, while the capital of North Korea is Pyongyang.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1764607014, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_b547601dbd', usage=CompletionUsage(completion_tokens=19, prompt_tokens=15, total_tokens=34, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [ ]:
# choice가 가장 중요한 이유는 ai가 생각한 답변 후보들이 모두 모여있는 곳 
# choice(~ , message = ~ 이게 답)
# 가장 첫 번째(0번) 답변 꺼내기
print(response.choices[0].message.content)

# 만약 n=10 이라서 10개를 다 보고 싶다면?
for item in response.choices:
    print(item.message.content)

In [ ]:
# json형태랑 비슷해서 get 가능할 것 같지만
# 실제 object라서 사용 불가 get(['message]) 불가. 객체는 점 찍어서 속성 꺼내야 함. data.message
message = response.choices[0].message.content

message

'The capital of South Korea is Seoul, while the capital of North Korea is Pyongyang.'

# 간단한 agent만들기
- ai agent란 사용자를 대신해서 어떤 행동을 해줄 수 있는 시스템
- 위 연습에서는 llm모델이 질문한 것만 대답해주는 형식
- agent는 모델한테 도구를 쥐어주는  것(코드 실행, 웹 브라우저 연결 등)

In [1]:
import openai

client = openai.OpenAI()

PROMPT =  '''
    I have the following functions in my system
    
    'get_weather'
    'get_currency'
    'get_news'
    
    # 규칙 설정
    이 모든 함수들은 나라 이름을 인자로 받음(예시 get_news(korea))
    그리고 내가 실행해야 할 함수 이름 알려줘
    
    그 외에는 아무 말도 하지말고, 함수 이름과 인자만 말해줘
    
    다음 질문에 대답해줘
    
    한국 날씨는 어때? 
    '''

response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages=[{"role":'user', 'content':PROMPT}],
)

response

ChatCompletion(id='chatcmpl-CijDMq7kK0JH8v7qq1g0EAZcWstad', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="get_weather('korea')", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1764776272, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_b547601dbd', usage=CompletionUsage(completion_tokens=6, prompt_tokens=110, total_tokens=116, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [2]:
message = response.choices[0].message.content

message

"get_weather('korea')"

### 이게 바로 **AI 에이전트(Agent)**의 핵심 원리입니다! AI는 똑똑한 "두뇌"지만, 인터넷 검색이나 실시간 날씨 확인을 직접 할 수 있는 "손발"이 없습니다. (GPT 모델 자체는 훈련된 데이터만 아니까요.)

그래서 이런 "이어달리기" 과정을 거치는 겁니다.

🏃‍♂️ 1단계: 주문 (User -> AI)

나: "지금 한국 날씨 어때?"

AI (두뇌): (음.. 난 실시간 날씨를 모르는데? 하지만 나한테 get_weather라는 도구가 있다는 건 알아.)

AI의 대답: "주인님! get_weather('Korea') 이 함수 좀 대신 실행해주세요!" (👈 지금 님이 여기까지 하신 거예요)

🏃‍♂️ 2단계: 실행 (User의 코드)

나(파이썬 코드): "어? AI가 이거 실행하라네?"

실행: 실제로 파이썬 코드가 기상청 API 등을 찔러서 날씨를 가져옴.

결과: "현재 기온 25도, 맑음"이라는 데이터를 얻음.

🏃‍♂️ 3단계: 보고 (User -> AI) ⭐ "돌려주는 단계"

나: (AI가 아까 시킨 거 해왔다고 알려줘야지)

나: "야 AI야! 아까 네가 시킨 get_weather 결과가 '25도, 맑음'이라는데?" (👈 이게 돌려주는 것!)

🏃‍♂️ 4단계: 최종 답변 (AI -> User)

AI: "아하! 정보 고마워."

AI의 최종 대답: "사용자님, 현재 한국의 날씨는 25도이며 맑다고 합니다."

요약하자면: 강의에서 말한 건 **"AI가 함수 이름을 말하면 -> 네가(코드가) 실제로 계산해서 -> 그 결과값을 다시 AI한테 알려줘야 -> AI가 문장으로 예쁘게 말해준다"**는 뜻입니다. 

AI의 사고 과정 (1초 만에 일어나는 일)
입력 (User): "한국 날씨는 어때?"

분석 (AI): "사용자가 '날씨'를 물어봤네? 내가 가진 도구 리스트(get_weather, get_currency, get_news) 중에 뭐가 필요하지?"

매칭: "아! get_weather가 필요하구나. 대상은 'Korea'고."

규칙 적용 (중요!): "근데 주인님(프롬프트)이 **'다른 잡담은 하지 말고 함수 이름이랑 인자만 말해'**라고 명령했지."

출력: get_weather('Korea')

💡 왜 이렇게 할까요? (이게 핵심!)
만약 규칙을 안 정해주면 AI는 이렇게 말할 겁니다.

"한국의 날씨를 확인해 드릴게요! 잠시만 기다려주세요..."

문제점: 이 문장을 파이썬 코드로 가져오면 에러가 납니다. 파이썬은 "한국의..."라는 코드를 실행할 줄 모르니까요.

해결책: 그래서 님께서 프롬프트로 **"파이썬이 바로 실행할 수 있는 깔끔한 명령어(get_weather)로만 뱉어!"**라고 강제한 것입니다.

즉, 지금 보시는 그 출력 메시지는 님에게 읽으라고 준 편지가 아니라, **다음 단계의 파이썬 코드에게 넘겨줄 "바통(Batton)"**입니다. 🏃‍♂️💨

# 위 코드는 에이전트가 이해하고 답을 출력하는 과정을 프롬프트로 길게길게 다 수동으로 설정하는 걸 이해하기 위한 과정
- 이것을 위해 할일 첫번째, 시스템에 메모리 활성화

In [ ]:
import openai
client = openai.OpenAI()

response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages = [{'role':'user', 'content': '안녕 내 이름은 태준이야'}],
)

print(response.choices[0].message.content)

안녕하세요, 태준님! 만나서 반갑습니다. 어떻게 도와드릴까요?


In [5]:
import openai
client = openai.OpenAI()

response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages = [{'role':'user', 'content': '내 이름 뭐게'}],
)

print(response.choices[0].message.content)

죄송하지만, 당신의 이름을 알 수 있는 방법이 없습니다. 하지만 당신의 이름을 알려주시면 반갑게 인사할 수 있어요!


In [3]:
# 위 결과에서 보듯 질문 사항에 대해, 이전 질문들은 기억하지 못함
# 매 질문할 때마다 이전 기억 다 리셋. gpt처럼 기존 입력 정보 기억하게 하려면 메모리 시스템 구축해야함
import openai

client = openai.OpenAI()
messages = []

In [1]:
# 대화 히스토리를 저장할 리스트 (이게 없으면 AI는 맨날 잊음)
messages = [] 

def call_ai():
    """
    OpenAI API를 호출하고, 응답을 받아 대화 기록(messages)에 추가하는 함수
    """
    # 1. API 호출: 지금까지 쌓인 대화 내용(messages)을 통째로 보냄
    response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages
    )
    
    # 2. 메시지 추출: 응답 객체에서 실제 텍스트 내용(content)만 꺼냄
    message = response.choices[0].message.content
    
    # 3. AI의 기억 추가: AI가 한 말도 기억해야 다음 대화가 이어지므로 리스트에 추가
    messages.append({
        'role': 'assistant', 
        'content': message 
    })
    
    # 4. 결과 출력
    print(f"AI: {message}")

In [4]:
# 아래 while문으로 사용자 입력받게 하기
# 보낸 메세지를 messages = []에 추가하도록
# --- 메인 실행 루프 (사용자 입력 대기) ---
while True:
    # 사용자로부터 텍스트 입력 받기
    message = input("LLM에게 질문할 것 입력")

    # 종료 조건: 'quit' 혹은 'q'를 입력하면 루프 탈출
    # 만약 대문자로 입력해도 다 입력되게 lower사용(찰떡같이 알아먹게)
    if message.lower() in ["quit", "q"]: 
        print("대화를 종료합니다.")
        break
    
    # 대화 계속 진행
    else:
        # 1. 사용자의 말 기억하기: 내가 방금 한 말을 messages 리스트에 추가
        messages.append({
            'role': 'user',
            'content': message
        })
        print(f'User: {message}') # 내가 입력한 질문 또는 문장
        # 2. AI 소환: 위에서 만든 함수를 실행해서 답장을 받아옴
        call_ai()
        # 이 함수 내에서 직접 선언해도 되고, 밖에서 선언한거 불러와도 됨
        # response = client.chat.completions.create(
        #     model = 'gpt-4o-mini',
        #     messages = messages
        # )

User: 내 이름은 태준이야
AI: 안녕하세요, 태준님! 만나서 반갑습니다. 어떻게 도와드릴까요?
User: 내 이름이 뭐라고?
AI: 당신의 이름은 태준이라고 하셨습니다. 맞나요?
User: 내 출신지는 한국이야
AI: 예, 한국에서 오신 태준님이군요! 한국의 어떤 지역 출신이신가요? 혹은 다른 궁금한 점이 있으신가요?
User: 내 출신지 어딘지 말해주고, 근처 가까운 나라 이름을 알려줘
AI: 태준님께서 한국 출신이라고 하셨으니, 한국의 출신지는 다양한 곳이 있을 수 있지만, 서울, 부산, 대구 등 여러 지역이 있습니다. 

한국과 가까운 나라로는 일본, 중국, 러시아 등이 있습니다. 일본은 특히 가까운 이웃나라로 쉽게 갈 수 있습니다. 어떤 내용이 더 궁금하신가요?
대화를 종료합니다.


In [ ]:
# 대화가 너무 길어지면, 너무 많으면.. 좀 그럼
# 그럴 땐 메세지를 압축하거나, 삭제하거나 그렇게 해야 함